In [1]:
import os
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# Define dataset path and transformations
dataset_path = 'dataset'
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load dataset
dataset = datasets.ImageFolder(root=dataset_path, transform=transform)

# Split dataset into training and validation sets
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)


## Model 1: Simple CNN

In [2]:
import torch.nn as nn

class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 53 * 53, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, len(dataset.classes))

    def forward(self, x):
        x = self.pool(nn.functional.relu(self.conv1(x)))
        x = self.pool(nn.functional.relu(self.conv2(x)))
        x = x.view(-1, 16 * 53 * 53)
        x = nn.functional.relu(self.fc1(x))
        x = nn.functional.relu(self.fc2(x))
        x = self.fc3(x)
        return x

model1 = SimpleCNN()
criterion1 = nn.CrossEntropyLoss()
optimizer1 = torch.optim.Adam(model1.parameters(), lr=0.001)

# Train model1
for epoch in range(10):
    for images, labels in train_loader:
        optimizer1.zero_grad()
        outputs = model1(images)
        loss = criterion1(outputs, labels)
        loss.backward()
        optimizer1.step()
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")


c:\Users\priya\miniconda3\envs\mmdp\Lib\site-packages\PIL\Image.py:1045: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch 1, Loss: 2.7174
Epoch 2, Loss: 2.6890
Epoch 3, Loss: 1.9634
Epoch 4, Loss: 1.2744
Epoch 5, Loss: 0.4182
Epoch 6, Loss: 0.1484
Epoch 7, Loss: 0.0472
Epoch 8, Loss: 0.0151
Epoch 9, Loss: 0.1088
Epoch 10, Loss: 0.0094


In [3]:
# test

correct = 0
total = 0
with torch.no_grad():
    for images, labels in val_loader:
        outputs = model1(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
print(f'Accuracy: {100 * correct / total}%')

Accuracy: 23.5%


## Model 2: Pre-trained ResNet18

In [6]:
from torchvision import models

model3 = models.resnet18(pretrained=True)
num_ftrs = model3.fc.in_features
model3.fc = nn.Linear(num_ftrs, len(dataset.classes))

criterion3 = nn.CrossEntropyLoss()
optimizer3 = torch.optim.Adam(model3.parameters(), lr=0.001)

# Train model3
for epoch in range(5):
    for images, labels in train_loader:
        optimizer3.zero_grad()
        outputs = model3(images)
        loss = criterion3(outputs, labels)
        loss.backward()
        optimizer3.step()
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")


Epoch 1, Loss: 1.2717
Epoch 2, Loss: 0.8140
Epoch 3, Loss: 0.6396
Epoch 4, Loss: 0.2447
Epoch 5, Loss: 0.9111


In [5]:
# test

correct = 0
total = 0
with torch.no_grad():
    for images, labels in val_loader:
        outputs = model1(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
print(f'Accuracy: {100 * correct / total}%')

Accuracy: 23.5%
